# DSE Baselines v2 — Notebook 3: DGL (`MVSL-DSF`) & Tổng hợp Bảng Bài báo Cuối cùng

Notebook này:
1. Cài đặt môi trường tương thích **DGL + DGLLife + RDKit** trên Google Colab hiện đại.
2. Chạy lại **`MVSL-DSF`** dưới Protocol v2 (Split B & Split A — 10 folds, dùng ma trận MedDRA ontology sạch `994×243` từ `side_effect_label_750.mat` và profile chỉ tính từ `train` 80%).
3. Tổng hợp **toàn bộ kết quả thực nghiệm cho bài báo** (tất cả Baselines + `MSSF_plus_full` + Ablations + Conformal C4 + Wilcoxon signed-rank test + bảng phụ lục `*_rmseSel_*`) từ `DRIVE_DIR`.

In [ ]:
# ===== 1. CLONE / UPDATE REPO & MOUNT GOOGLE DRIVE =====
import os, subprocess, torch

REPO_URL = 'https://github.com/hungbuile04/DSE.git'
REPO_BRANCH = 'main'
WORK_DIR = '/content/DSE'

if not os.path.exists(WORK_DIR):
    subprocess.run(['git', 'clone', '-b', REPO_BRANCH, REPO_URL, WORK_DIR], check=True)
else:
    subprocess.run(['git', '-C', WORK_DIR, 'pull'], check=False)

os.chdir(WORK_DIR)
print('Working directory:', os.getcwd())

# Mount Google Drive for per-fold auto-backup & seamless resume
DRIVE_DIR = '/content/drive/MyDrive/DSE_results_v2'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print(f'Google Drive mounted. Auto-sync directory: {DRIVE_DIR}')
except Exception as e:
    DRIVE_DIR = '/content/DSE_drive_fallback'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print(f'Google Drive not mounted ({e}); using local fallback: {DRIVE_DIR}')

os.environ['DRIVE_DIR'] = DRIVE_DIR

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    try:
        torch.set_float32_matmul_precision('medium')
    except Exception:
        pass
    print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)')
else:
    print('WARNING: CUDA GPU not detected! Enable Runtime -> Change runtime type -> GPU.')

# ===== Install RDKit, DGLLife, and a compatible DGL wheel =====
import sys, subprocess, torch

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rdkit', 'dgllife', 'pydantic', 'torchdata'], check=False)

# Try importing DGL; if missing or ABI-incompatible with Colab's default Torch, install matching wheel
def _ensure_dgl():
    try:
        import dgl
        print('DGL already working:', dgl.__version__)
        return
    except Exception:
        pass

    torch_ver = torch.__version__.split('+')[0]
    major_minor = '.'.join(torch_ver.split('.')[:2])
    cu_tag = ('cu' + torch.version.cuda.replace('.', '')) if torch.cuda.is_available() and torch.version.cuda else 'cpu'
    candidates = [
        f'https://data.dgl.ai/wheels/torch-{major_minor}/{cu_tag}/repo.html',
        'https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html',
        'https://data.dgl.ai/wheels/torch-2.4/cu121/repo.html',
        'https://data.dgl.ai/wheels/torch-2.3/cu121/repo.html',
    ]
    for url in candidates:
        print(f'Trying DGL wheel index: {url}')
        res = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'dgl', '-f', url], check=False)
        if res.returncode == 0:
            break

_ensure_dgl()

# MVSL-DSF only uses core DGL message passing (not C++ GraphBolt). If Colab runs PyTorch >= 2.5
# where libgraphbolt_pytorch_<ver>.so is absent, disable the optional graphbolt import automatically:
try:
    import dgl
    print('Verified DGL:', dgl.__version__)
except FileNotFoundError as e:
    if 'graphbolt' in str(e):
        import importlib.util
        spec = importlib.util.find_spec('dgl')
        gb_init = os.path.join(os.path.dirname(spec.origin), 'graphbolt', '__init__.py')
        if os.path.exists(gb_init):
            with open(gb_init, 'w', encoding='utf-8') as f:
                f.write('# Stubbed optional graphbolt for PyTorch >= 2.5 compatibility
')
        import dgl
        print('Verified DGL (with GraphBolt stub for PyTorch >= 2.5):', dgl.__version__)
    else:
        raise

In [ ]:
# ===== 2. VERIFY BENCHMARK DATA INTEGRITY (SHA256) =====
!python MSSF_plus/audit/verify_checksums.py --check

---
## 3. Baseline 7: `MVSL-DSF` (Split B & Split A — 10 Folds)

In [ ]:
%%time
!python -m MSSF_plus.baselines_rerun.run_mvsl \
    --split B \
    --mode full \
    --se_source mat \
    --results_dir ./results_v2 \
    --drive_dir "$DRIVE_DIR" \
    --resume

!python -m MSSF_plus.baselines_rerun.run_mvsl \
    --split A \
    --mode full \
    --se_source mat \
    --results_dir ./results_v2 \
    --drive_dir "$DRIVE_DIR" \
    --resume

---
## 4. Xuất Toàn Bộ Bảng Kết Quả Cuối Cùng Cho Bài Báo (Master Paper Tables)
Đọc toàn bộ kết quả đã đồng bộ trong `DRIVE_DIR` từ cả 4 Notebook (`colab_0` .. `colab_3`) và xuất:
1. `MASTER_PAPER_TABLES_V2.md` (5-class Classification, Ordinal QWK/MAE, Per-class F1 `c0..c4`, Conformal C4, kiểm định Wilcoxon signed-rank `p`-value).
2. `MASTER_UNIFIED_EVAL_V2.csv` (Đầy đủ cả chỉ số Regression `RMSE, MAE, Pearson, Spearman` lẫn 5-class metrics cho bảng chính & phụ lục `*_rmseSel_*`).

In [ ]:
!python -m MSSF_plus.scripts.summarize_results \
    --results_dir "$DRIVE_DIR" \
    --ref_method MSSF_v2 \
    --out_md "$DRIVE_DIR/MASTER_PAPER_TABLES_V2.md"

!python shared_data/unified_eval.py \
    --results_dir "$DRIVE_DIR" \
    --csv "$DRIVE_DIR/MASTER_UNIFIED_EVAL_V2.csv"

print('\nSaved publication reports to:')
print(f'  - {DRIVE_DIR}/MASTER_PAPER_TABLES_V2.md')
print(f'  - {DRIVE_DIR}/MASTER_UNIFIED_EVAL_V2.csv')